# Comment Classifier

In [1]:

from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

load_dotenv()

from pydantic import BaseModel, Field 
from typing import TypedDict, List 


class TopicState(TypedDict, total=False):
    """State passed between LangGraph nodes."""

    comments: List[str]
    topics: List[str]
    user_topics : List[str]
    classified_comments: List[dict]
    summary: List[dict]
    generate_topic_summary: bool


class TopicDiscoveryOutput(BaseModel):
    """LLM output containing discovered discussion topics."""

    topics: List[str] = Field(
        description="Short, non-overlapping topic names"
    )


class ClassifiedComment(BaseModel):
    """Single classified comment with its assigned topic."""

    comment: str
    topic: str

class TopicClassificationOutput(BaseModel):
    """LLM output containing classification results for all comments."""

    results: List[ClassifiedComment]

class TopicSummary(BaseModel):
    """Schema for generating a summary of comments grouped by topic."""
    
    topic: str
    summary: str = Field(
        description="Generate a concise summary (maximum 5 lines) that captures the key points, common opinions, and main discussion themes from the comments related to this topic."
    )
    
class TopicSummaryOutput(BaseModel):
    "list of Output for multiple topics"
    final_summary : List[TopicSummary]

In [14]:
class TopicClassifier:

    def __init__(self, api_key: str,min_comment_count_for_summary: int = 4 ,  model_name: str = "gemini-2.5-flash", temperature: float = 0.0):
        """
        Initialize the LLM and structured output parsers
        """
        self.api_key = api_key
        self.model_name = model_name
        self.temperature = temperature
        self.min_comment_count_for_summary = min_comment_count_for_summary
        # LLM with API key
        self.llm = ChatGoogleGenerativeAI(model=self.model_name,
                                          temperature=self.temperature,
                                          api_key=self.api_key)

        # structured outputs
        self.topic_discovery_llm = self.llm.with_structured_output(TopicDiscoveryOutput)
        self.topic_classification_llm = self.llm.with_structured_output(TopicClassificationOutput)

    def _discover_topics(self, state: TopicState):
        """
        Discover discussion topics from comments OR use user-provided topics
        """

        # If user manually provides topics → use them
        if state.get("user_topics") and len(state["user_topics"]) > 0:
            return {
                **state,
                "topics": state["user_topics"]
            }

        # Otherwise discover topics using LLM
        comments_text = "\n".join(state["comments"][:100])

        prompt = f"""
        You are analyzing YouTube comments.

        TASK:
        - Create discussion topics from the comments
        - Topics must be created by you
        - 2–3 words per topic
        - No sentiment words
        - Max 8 topics
        - Dont include "Other" as a topic

        COMMENTS:
        {comments_text}
        """

        response = self.topic_discovery_llm.invoke(prompt)

        return {
            **state,
            "topics": response.topics
        }

    def _classify_comments(self, state: TopicState):
        """
        Classify comments into discovered topics
        """

        comments_text = "\n".join(state["comments"])
        topics_text = ", ".join(state["topics"])

        prompt = f"""
        You are classifying comments into topics.

        TOPICS:
        {topics_text}

        RULES:
        - Use ONLY the provided topics
        - One topic per comment
        - Do not invent new topics
        - No explanations

        COMMENTS:
        {comments_text}
        """

        response = self.topic_classification_llm.invoke(prompt)

        return {
            **state,
            "classified_comments": [
                item.model_dump() for item in response.results
            ]
        }
    def topics_summary(self, state: TopicState):
        """
        Generate summaries for each discovered topic based on grouped comments
        """

        topic_comments = {}

        for item in state['classified_comments']:
            topic = item["topic"]
            comment = item["comment"]

            if topic not in topic_comments:
                topic_comments[topic] = []

            topic_comments[topic].append(comment)

        
        
        prompt_parts = []

        for topic, comments in topic_comments.items():
            if len(comments)>=self.min_comment_count_for_summary:
                comments_text = "\n".join(comments)

                prompt_parts.append(f"""
                TOPIC: {topic}

                COMMENTS:
                {comments_text}
                """)

        prompt = f"""
        You are analyzing YouTube comments grouped by discussion topic.

        TASK:
        Generate a short analytical summary for each topic.

        RULES:
        - Maximum 5 lines per topic
        - Focus on key insights, doubts, and common opinions
        - Do not repeat comments
        - Keep summaries concise and analytical

        TOPIC DATA:
        {''.join(prompt_parts)}
        """

        summary_llm = self.llm.with_structured_output(TopicSummaryOutput)

        response = summary_llm.invoke(prompt)

        return {
            **state,
            "summary": [
                item.model_dump() for item in response.final_summary
            ]
        }
        
    def summary_router(self, state: TopicState):
    
        if state.get("generate_topic_summary", False):
            return "topics_summary"
        
        return END
    def graph(self):

        graph = StateGraph(TopicState)

        graph.add_node("discover_topics", self._discover_topics)
        graph.add_node("classify_comments", self._classify_comments)
        graph.add_node("topics_summary", self.topics_summary)

        graph.add_edge(START, "discover_topics")
        graph.add_edge("discover_topics", "classify_comments")

        graph.add_conditional_edges(
            "classify_comments",
            self.summary_router
        )

        graph.add_edge("topics_summary", END)

        topic_graph = graph.compile()

        return topic_graph

In [15]:
comments = {
    "comments": [
        {
            "text": "Great explanation of backpropagation in neural networks, it finally makes sense",
            "timestamp": "2026-02-01T02:35:03Z",
            "authorId": "user1"
        },
        {
            "text": "Can someone explain the difference between transformers and traditional RNNs?",
            "timestamp": "2026-01-31T18:59:54Z",
            "authorId": "user2"
        },
        {
            "text": "These deep learning notes on CNN architectures are very clear and helpful",
            "timestamp": "2026-01-30T14:22:11Z",
            "authorId": "user3"
        },
        {
            "text": "I am confused about how attention mechanism actually works inside LLMs",
            "timestamp": "2026-01-29T09:18:45Z",
            "authorId": "user4"
        },
        {
            "text": "The explanation of gradient descent and learning rate scheduling is really useful",
            "timestamp": "2026-01-28T21:40:12Z",
            "authorId": "user5"
        },
        {
            "text": "Are embeddings in LLMs similar to word2vec embeddings or are they different?",
            "timestamp": "2026-01-27T16:33:08Z",
            "authorId": "user6"
        },
        {
            "text": "These notes helped me understand how transformers process tokens in parallel",
            "timestamp": "2026-01-26T12:10:19Z",
            "authorId": "user7"
        },
        {
            "text": "I still have a doubt about how positional encoding works in transformers",
            "timestamp": "2026-01-25T08:55:42Z",
            "authorId": "user8"
        },
        {
            "text": "The section on fine tuning LLMs with LoRA was very insightful",
            "timestamp": "2026-01-24T19:27:50Z",
            "authorId": "user9"
        },
        {
            "text": "Can anyone clarify how RAG improves LLM responses compared to normal prompting?",
            "timestamp": "2026-01-23T11:42:31Z",
            "authorId": "user10"
        }
    ]
}

In [16]:
comments = [c['text'] for c in comments["comments"]]

In [21]:
api_key = os.getenv("GOOGLE_API_KEY")
classifier = TopicClassifier(api_key=api_key)

# 2️⃣ Compile graph
topic_graph = classifier.graph()

# 3️⃣ Prepare input state
input_state = {
    "comments": comments,
    
    # Optional user topics
    "user_topics": ["neural networks" ,"Doubts"],

    # decide if summary should run
    "generate_topic_summary": False
}

# 4️⃣ Run graph
result = topic_graph.invoke(input_state)

print(result)

{'comments': ['Great explanation of backpropagation in neural networks, it finally makes sense', 'Can someone explain the difference between transformers and traditional RNNs?', 'These deep learning notes on CNN architectures are very clear and helpful', 'I am confused about how attention mechanism actually works inside LLMs', 'The explanation of gradient descent and learning rate scheduling is really useful', 'Are embeddings in LLMs similar to word2vec embeddings or are they different?', 'These notes helped me understand how transformers process tokens in parallel', 'I still have a doubt about how positional encoding works in transformers', 'The section on fine tuning LLMs with LoRA was very insightful', 'Can anyone clarify how RAG improves LLM responses compared to normal prompting?'], 'topics': ['neural networks', 'Doubts'], 'user_topics': ['neural networks', 'Doubts'], 'classified_comments': [{'comment': 'Great explanation of backpropagation in neural networks, it finally makes sen

In [26]:
result['classified_comments']

[{'comment': 'Great explanation of backpropagation in neural networks, it finally makes sense',
  'topic': 'neural networks'},
 {'comment': 'Can someone explain the difference between transformers and traditional RNNs?',
  'topic': 'Doubts'},
 {'comment': 'These deep learning notes on CNN architectures are very clear and helpful',
  'topic': 'neural networks'},
 {'comment': 'I am confused about how attention mechanism actually works inside LLMs',
  'topic': 'Doubts'},
 {'comment': 'The explanation of gradient descent and learning rate scheduling is really useful',
  'topic': 'neural networks'},
 {'comment': 'Are embeddings in LLMs similar to word2vec embeddings or are they different?',
  'topic': 'Doubts'},
 {'comment': 'These notes helped me understand how transformers process tokens in parallel',
  'topic': 'neural networks'},
 {'comment': 'I still have a doubt about how positional encoding works in transformers',
  'topic': 'Doubts'},
 {'comment': 'The section on fine tuning LLMs wi

In [20]:
topic_comments = {}

for item in result['classified_comments']:
    topic = item["topic"]
    comment = item["comment"]

    if topic not in topic_comments:
        topic_comments[topic] = []

    topic_comments[topic].append(comment)

topic_comments

{'neural networks': ['Great explanation of backpropagation in neural networks, it finally makes sense',
  'These deep learning notes on CNN architectures are very clear and helpful',
  'The explanation of gradient descent and learning rate scheduling is really useful'],
 'Doubts': ['Can someone explain the difference between transformers and traditional RNNs?',
  'I am confused about how attention mechanism actually works inside LLMs',
  'Are embeddings in LLMs similar to word2vec embeddings or are they different?',
  'I still have a doubt about how positional encoding works in transformers',
  'Can anyone clarify how RAG improves LLM responses compared to normal prompting?'],
 'LLMS': ['These notes helped me understand how transformers process tokens in parallel',
  'The section on fine tuning LLMs with LoRA was very insightful']}

In [55]:
prompt_parts = []

for topic, comments in topic_comments.items():
    if len(comments)>=3:
        comments_text = "\n".join(comments)

        prompt_parts.append(f"""
        TOPIC: {topic}

        COMMENTS:
        {comments_text}

        """)
prompt_parts

['\n        TOPIC: Transformer Architecture\n\n        COMMENTS:\n        Can someone explain the difference between transformers and traditional RNNs?\nThese notes helped me understand how transformers process tokens in parallel\nI still have a doubt about how positional encoding works in transformers\n\n        ']

In [58]:
agent = TopicClassifier(api_key=api_key)
summary = agent.topics_summary(state=result , min_comment_count_for_summary=3)

In [59]:
summary['topic_summaries']

[{'topic': 'Transformer Architecture',
  'summary': 'The discussion highlights a strong interest in understanding Transformer architecture, particularly its key differences from traditional RNNs. Users appreciate the concept of parallel token processing as a significant advantage. However, a common point of confusion or doubt remains regarding the detailed mechanism and function of positional encoding within the Transformer model.'}]

## manual run

In [10]:
state = {
    "comments": comments,
    "generate_topic_summary": True,
    "user_topics" : ["neural nets" , "LLMs" , "doubts"]
}

In [11]:
state = classifier._discover_topics(state)
print(state["topics"])

['neural nets', 'LLMs', 'doubts']


In [12]:
state = classifier._classify_comments(state)
print(state["classified_comments"])

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED'}}